# Pipeline RAG con LangChain

Este notebook implementa paso a paso un pipeline **Retrieval-Augmented Generation (RAG)**:
carga de documentos → chunking → embeddings → base vectorial Chroma → retriever → cadena de preguntas y respuestas con OpenAI.

**Requisitos:** Python 3.10+, entorno virtual recomendado y una `OPENAI_API_KEY` válida.

Usa el kernel del entorno del proyecto: `../.venv/bin/python` (tras crear el venv e instalar dependencias).

## 1. Instalar paquetes necesarios

Cada paquete cumple un rol distinto en el pipeline:

| Paquete | Función |
|---------|--------|
| `langchain` | Framework principal para encadenar componentes |
| `langchain-community` | Document loaders (TextLoader, PyPDFLoader, etc.) |
| `langchain-openai` | Conexión con la API de OpenAI (embeddings y chat) |
| `langchain-chroma` | Base de datos vectorial Chroma |
| `pypdf` | Lectura de archivos PDF |
| `langchain-classic` | Cadenas clásicas como `RetrievalQA` (LangChain 1.x) |
| `langchain-text-splitters` | Utilidades de división de texto (LangChain 1.x) |

In [ ]:
# Ejecuta esta celda una vez por entorno (o instala desde la terminal con el venv activo).
%pip install -q langchain langchain-community langchain-openai langchain-chroma pypdf langchain-classic langchain-text-splitters

## 2. Importar herramientas

Agrupamos las importaciones según su responsabilidad en el pipeline.

In [ ]:
# Document loaders: TextLoader lee .txt; PyPDFLoader extrae texto de PDFs.
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# División en chunks. En LangChain 1.x vive en langchain_text_splitters
# (en material antiguo: from langchain.text_splitter import RecursiveCharacterTextSplitter).
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings y modelo de chat de OpenAI.
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Base vectorial Chroma.
from langchain_chroma import Chroma

# Cadena RAG preconstruida: retrieval + question answering.
# En LangChain 1.x: from langchain_classic.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA

# Variables de entorno (p. ej. OPENAI_API_KEY).
import os
from pathlib import Path

## 3. Configurar la API key de OpenAI

Los componentes `OpenAIEmbeddings` y `ChatOpenAI` leen la clave desde la variable de entorno `OPENAI_API_KEY`.

Opciones:
1. Exportarla en la terminal antes de abrir Jupyter.
2. Asignarla en la celda siguiente (no subas la clave a git).
3. Usar un archivo `.env` con `python-dotenv` si ya lo tienes en el proyecto.

In [ ]:
# Descomenta y reemplaza solo si NO tienes OPENAI_API_KEY en el entorno:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "Define OPENAI_API_KEY en el entorno o descomenta la línea de asignación en esta celda."
    )

## 4. Cargar documentos (TXT o PDF)

Indica la ruta de tu archivo en `document_path` (`.txt` o `.pdf`). El **router** elige automáticamente `TextLoader` o `PyPDFLoader` según la extensión.

También puedes pasar varias rutas en `document_paths` para indexar varios archivos en un solo pipeline.

In [ ]:
def load_documents_from_path(path: Path) -> list:
    """Routing por extensión: .pdf → PyPDFLoader, .txt/.md → TextLoader."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el archivo: {path.resolve()}")

    suffix = path.suffix.lower()
    if suffix == ".pdf":
        loader = PyPDFLoader(str(path))
    elif suffix in {".txt", ".md"}:
        loader = TextLoader(str(path), encoding="utf-8")
    else:
        raise ValueError(
            f"Extensión no soportada: {suffix!r}. Usa .pdf, .txt o .md."
        )

    return loader.load()


def load_documents_from_paths(paths: list[Path]) -> list:
    documents = []
    for path in paths:
        loaded = load_documents_from_path(path)
        documents.extend(loaded)
        print(
            f"  {path.name} ({path.suffix.lower()}) → "
            f"{len(loaded)} documento(s) LangChain"
        )
    return documents


# --- Configura aquí tu(s) archivo(s) ---
document_path = Path("../data/sample_document.txt")
# document_path = Path("../data/mi_documento.pdf")

paths_to_load = [document_path]
# paths_to_load = [
#     Path("../data/sample_document.txt"),
#     Path("../data/informe.pdf"),
# ]

print("Cargando:")
documents = load_documents_from_paths(paths_to_load)

total_chars = sum(len(doc.page_content) for doc in documents)
print(f"\nTotal objetos Document: {len(documents)}")
print(f"Caracteres totales (aprox.): {total_chars}")
if documents:
    preview = documents[0].page_content[:200].replace("\n", " ")
    print(f"Preview: {preview}...")

## 5. Dividir en chunks

Los LLM tienen límites de contexto y el retrieval funciona mejor con fragmentos enfocados.
`chunk_overlap` mantiene continuidad entre chunks consecutivos.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)
print(f"Número de chunks: {len(chunks)}")
print("--- Primer chunk (preview) ---")
print(chunks[0].page_content[:300], "...")

## 6. Embeddings y vector store (Chroma)

`OpenAIEmbeddings` convierte cada chunk en un vector.
Chroma indexa esos vectores y puede persistir en disco para no reindexar en cada ejecución.

In [ ]:
embeddings = OpenAIEmbeddings()

persist_dir = "../chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir,
)

print(f"Vector store creado. Persistencia en: {persist_dir}")

## 7. Retriever

El retriever convierte la pregunta en embedding y devuelve los `k` chunks más similares (`search_type="similarity"`).

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Vista previa opcional: qué chunks recuperaría una consulta.
preview_query = "¿Qué es RAG?"
preview_docs = retriever.invoke(preview_query)
print(f"Preview retrieval para: {preview_query!r}")
for i, doc in enumerate(preview_docs, start=1):
    print(f"\n[{i}] {doc.page_content[:200]}...")

## 8. Modelo de lenguaje (LLM)

`ChatOpenAI` genera la respuesta final. Ajusta `model` y `temperature` según el comportamiento deseado
(temperatura más alta → respuestas más variadas).

In [ ]:
# El material del curso usa model_name="gpt-5-nano"; en langchain-openai reciente el parámetro es model=.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

print("LLM configurado:", llm.model_name)

## 9. Cadena RetrievalQA (RAG end-to-end)

`chain_type="stuff"` inserta todos los chunks recuperados en el prompt.
`return_source_documents=True` devuelve también los fragmentos usados (provenance).

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("Cadena RAG lista.")

## 10. Consulta y resultados

Flujo completo: embedding de la pregunta → búsqueda en Chroma → construcción del prompt → respuesta del LLM.

In [ ]:
query = "¿De qué trata este documento?"

# invoke() es la forma recomendada; también puedes usar qa_chain({"query": query}).
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSource documents:", len(result["source_documents"]))
for i, doc in enumerate(result["source_documents"], start=1):
    print(f"\n--- Fuente {i} ---")
    print(doc.page_content[:400], "..." if len(doc.page_content) > 400 else "")